# Similaridade entre documentos e versões

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/04_similaridade_documentos_e_versoes_revisado.ipynb)

O Notebook 03 transformou textos em conjuntos de tokens, vetores de contagem e vetores TF-IDF. Agora podemos comparar documentos, mas é preciso começar por uma ideia fundamental:

> **“semelhante” não é uma propriedade única de dois textos. A resposta depende da representação e da medida escolhidas.**

Dois documentos podem compartilhar grande parte do vocabulário e, ainda assim, usar esses termos com frequências muito diferentes. Também podem apresentar perfis de frequência parecidos, mas divergir justamente nos termos raros que distinguem seus temas.

Neste notebook trabalharemos com três medidas centrais:

1. **Jaccard**, para presença e ausência de termos;
2. **cosseno sobre contagens**, para perfis de frequência;
3. **cosseno sobre TF-IDF**, para perfis ponderados por raridade na coleção.

Ao final, introduziremos a distância de Levenshtein como medida específica para comparar **versões de um mesmo texto**, situação em que a ordem das unidades volta a ser importante.

## 1. Escolher a medida pela pergunta

| Pergunta | Representação | Medida inicial |
|---|---|---|
| Compartilham vocabulário? | conjunto de termos | Jaccard |
| Têm perfis de frequência parecidos? | vetor de contagens | cosseno |
| Compartilham termos distintivos? | vetor TF-IDF | cosseno |
| Quanto uma versão precisa mudar? | sequência de tokens | distância de edição |

A escolha da métrica deve vir **depois da pergunta e da representação**, não depois de observar qual resultado parece mais interessante.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import os
import subprocess

URL_REPOSITORIO = "https://github.com/correa-ufrrj/disciplina_computacao_aplicada_humanidades_digitais.git"
REPOSITORIO = Path("/dados/disciplina_computacao_aplicada_humanidades_digitais")
PASTA_UNIDADE = REPOSITORIO / "unidade_05"

try:
    import google.colab  # type: ignore
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             URL_REPOSITORIO, str(REPOSITORIO)],
            check=True,
        )
    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

### Reconstrução da mesma base textual do Notebook 03

Usaremos os mesmos **120 documentos fictícios** do conjunto ampliado da Unidade 5.

Como no Notebook 03, cada registro recebe um texto sintético construído a partir de `tema`, `genero`, `local` e período. Em seguida aplicamos exatamente a mesma tokenização e reconstruímos:

- a matriz documento-termo;
- a matriz TF-IDF.

Essa reconstrução é importante porque as medidas de similaridade só são comparáveis quando usam o **mesmo vocabulário, o mesmo pré-processamento e a mesma ponderação**.

In [ ]:
import math
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ARQUIVO = Path("dados/documentos.csv")

def gerar_dados():
    SEED, N = 20261058, 120
    rng = np.random.default_rng(SEED)

    linhas = []
    for ano in range(1890, 1910):
        t = (ano - 1890) / 19
        p_capital = 0.12 + 0.76 * t
        n_capital = int(np.clip(rng.binomial(6, p_capital), 1, 5))
        locais_ano = np.array(
            ["Capital"] * n_capital + ["Interior"] * (6 - n_capital)
        )
        rng.shuffle(locais_ano)
        linhas.extend((ano, local) for local in locais_ano)

    rng.shuffle(linhas)
    anos = np.array([x[0] for x in linhas])
    locais = np.array([x[1] for x in linhas])
    capital = locais == "Capital"

    generos = rng.choice(
        ["notícia", "editorial", "carta"],
        N,
        p=[0.45, 0.30, 0.25],
    )

    temas_possiveis = np.array(
        ["educação", "trabalho", "progresso", "saúde"]
    )
    probs = {
        "notícia": [0.20, 0.25, 0.15, 0.40],
        "editorial": [0.20, 0.30, 0.35, 0.15],
        "carta": [0.40, 0.25, 0.20, 0.15],
    }
    temas = np.array([
        rng.choice(temas_possiveis, p=probs[g])
        for g in generos
    ])

    efeito_genero = {
        "notícia": 0.03,
        "editorial": 0.24,
        "carta": -0.18,
    }
    log_palavras = (
        math.log(650)
        + np.array([efeito_genero[g] for g in generos])
        + np.where(capital, 0.64, 0.0)
        - 0.016 * (anos - 1890)
        + rng.normal(0, 0.24, N)
    )
    palavras = np.maximum(
        140, np.rint(np.exp(log_palavras))
    ).astype(int)

    especiais = rng.choice(N, 3, replace=False)
    palavras[especiais] = np.rint(
        palavras[especiais] * rng.uniform(1.65, 2.05, 3)
    ).astype(int)

    densidade = np.clip(rng.normal(365, 38, N), 270, 470)
    paginas = np.maximum(
        1, np.rint(palavras / densidade)
    ).astype(int)

    taxa = {
        "notícia": 7.0,
        "editorial": 4.7,
        "carta": 5.8,
    }
    pessoas = rng.poisson(
        palavras / 1000 * np.array([taxa[g] for g in generos])
    )

    df = pd.DataFrame({
        "id_documento": [f"R{i:03d}" for i in range(1, N + 1)],
        "ano": anos,
        "genero": generos,
        "local": locais,
        "tema": temas,
        "palavras": palavras,
        "paginas": paginas,
        "pessoas": pessoas,
    })
    ARQUIVO.parent.mkdir(exist_ok=True)
    df.to_csv(ARQUIVO, index=False)
    return df

if ARQUIVO.exists():
    dados = pd.read_csv(ARQUIVO)
    print("Dados carregados de:", ARQUIVO)
else:
    dados = gerar_dados()
    print("Dados reproduzidos em:", ARQUIVO)

vocabulario_comum = [
    "debate", "sociedade", "questão", "público",
    "mudança", "vida", "cidade", "discussão",
]

vocabulario_tema = {
    "educação": [
        "escola", "ensino", "professor", "aluno",
        "instrução", "leitura", "aprendizagem",
    ],
    "trabalho": [
        "trabalho", "ofício", "operário", "salário",
        "jornada", "produção", "emprego",
    ],
    "progresso": [
        "progresso", "modernização", "indústria",
        "técnica", "desenvolvimento", "inovação", "futuro",
    ],
    "saúde": [
        "saúde", "doença", "higiene", "hospital",
        "médico", "prevenção", "tratamento",
    ],
}

vocabulario_genero = {
    "notícia": ["informação", "ocorreu", "registro", "relato"],
    "editorial": ["opinião", "argumento", "defesa", "posição"],
    "carta": ["escrevo", "pedido", "lembrança", "correspondência"],
}

vocabulario_local = {
    "Capital": ["avenida", "centro", "imprensa", "instituição"],
    "Interior": ["distrito", "comunidade", "região", "localidade"],
}

vocabulario_periodo = {
    "inicial": ["ordem", "costume", "reforma"],
    "final": ["transformação", "novidade", "expansão"],
}

def construir_texto(linha):
    periodo = "inicial" if linha["ano"] <= 1899 else "final"
    termos = (
        vocabulario_comum[:4]
        + vocabulario_tema[linha["tema"]]
        + vocabulario_genero[linha["genero"]]
        + vocabulario_local[linha["local"]]
        + vocabulario_periodo[periodo]
    )
    termos += vocabulario_tema[linha["tema"]][:3]
    termos += vocabulario_comum[4:7]
    return " ".join(termos)

dados["texto"] = dados.apply(construir_texto, axis=1)

stopwords = {
    "a", "e", "o", "de", "do", "da",
    "em", "nas", "também",
}

def tokenizar(texto):
    base = unicodedata.normalize("NFKD", texto.lower())
    base = "".join(
        c for c in base
        if not unicodedata.combining(c)
    )
    return [
        t for t in re.findall(r"[a-z]+", base)
        if t not in stopwords
    ]

dados["tokens"] = dados["texto"].map(tokenizar)

matriz_contagens = pd.DataFrame(
    [Counter(tokens) for tokens in dados["tokens"]],
    index=dados["id_documento"],
).fillna(0).astype(int)

matriz_contagens.index.name = "id_documento"

N = len(matriz_contagens)
df = matriz_contagens.gt(0).sum(axis=0)
idf = ((N + 1) / (df + 1)).map(math.log) + 1
matriz_tfidf = matriz_contagens.mul(idf, axis=1)

print("Documentos:", len(dados))
print("Termos no vocabulário:", matriz_contagens.shape[1])

## 2. Similaridade de Jaccard

A primeira pergunta é:

> **Dois documentos compartilham vocabulário?**

Para respondê-la, ignoramos quantas vezes cada termo aparece e consideramos apenas o **conjunto de termos presentes**.

Para dois conjuntos $A$ e $B$:

$$
J(A,B)
=
\frac{|A\cap B|}{|A\cup B|}.
$$

O numerador conta os termos compartilhados; o denominador conta todos os termos que aparecem em pelo menos um dos documentos.

A similaridade varia entre 0 e 1:

- $J=0$: nenhum termo compartilhado;
- $J=1$: os conjuntos de termos são idênticos.

Repetir uma palavra muitas vezes **não altera** Jaccard. Essa propriedade é adequada quando a pergunta se refere à presença de vocabulário, mas inadequada quando a frequência das palavras é relevante.

In [ ]:
def jaccard(tokens_a, tokens_b):
    a, b = set(tokens_a), set(tokens_b)
    uniao = a | b
    return len(a & b) / len(uniao) if uniao else 1.0

# Escolhemos automaticamente um documento de consulta.
consulta = dados.loc[
    dados["tema"].eq("educação"),
    "id_documento",
].iloc[0]

dados_indexados = dados.set_index("id_documento")
tokens_consulta = dados_indexados.loc[consulta, "tokens"]

similaridades_jaccard = (
    dados_indexados["tokens"]
    .map(lambda tokens: jaccard(tokens_consulta, tokens))
    .drop(consulta)
    .sort_values(ascending=False)
)

print("Documento-consulta:", consulta)
display(
    dados_indexados.loc[
        [consulta],
        ["ano", "genero", "local", "tema", "texto"],
    ]
)
display(
    similaridades_jaccard.head(8).rename("Jaccard")
)

O ranking mostra os documentos cujo **conjunto de termos** mais se aproxima do documento-consulta.

Mas todos os termos presentes recebem o mesmo peso. Um termo que aparece uma vez e outro que aparece quatro vezes contribuem igualmente para Jaccard.

Para incorporar frequências, precisamos voltar aos vetores.

## 3. Similaridade de cosseno

Para vetores $\mathbf{x}$ e $\mathbf{y}$, a similaridade de cosseno é:

$$
\cos(\mathbf{x},\mathbf{y})
=
\frac{
\mathbf{x}\cdot\mathbf{y}
}{
\|\mathbf{x}\|_2\|\mathbf{y}\|_2
}.
$$

Ela compara a **direção** dos vetores. Em vetores de contagens ou TF-IDF, cujos componentes são não negativos:

- valores próximos de 1 indicam perfis semelhantes;
- valores próximos de 0 indicam pouco compartilhamento entre componentes.

Como a medida é normalizada pelos comprimentos dos vetores, dois documentos de tamanhos diferentes podem apresentar cosseno alto quando possuem proporções lexicais parecidas.

Ainda assim, o resultado depende da representação utilizada. O cosseno sobre contagens e o cosseno sobre TF-IDF respondem a perguntas diferentes.

In [ ]:
def cosseno(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    denominador = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denominador) if denominador else 0.0

vetor_contagem = matriz_contagens.loc[consulta]
vetor_tfidf = matriz_tfidf.loc[consulta]

similaridades_cosseno_contagem = (
    matriz_contagens
    .drop(index=consulta)
    .apply(
        lambda linha: cosseno(vetor_contagem, linha),
        axis=1,
    )
    .sort_values(ascending=False)
)

similaridades_cosseno_tfidf = (
    matriz_tfidf
    .drop(index=consulta)
    .apply(
        lambda linha: cosseno(vetor_tfidf, linha),
        axis=1,
    )
    .sort_values(ascending=False)
)

ranking = pd.concat([
    similaridades_jaccard.rename("Jaccard"),
    similaridades_cosseno_contagem.rename("cosseno contagens"),
    similaridades_cosseno_tfidf.rename("cosseno TF-IDF"),
], axis=1)

display(
    ranking.sort_values(
        "cosseno TF-IDF",
        ascending=False,
    ).head(12).round(3)
)

### Por que os rankings podem divergir?

As três medidas atribuem importância diferente às propriedades do texto:

- **Jaccard**: considera apenas presença e ausência;
- **cosseno de contagens**: considera frequências;
- **cosseno TF-IDF**: considera frequências e reduz o peso dos termos comuns à coleção.

Por isso, não devemos esperar que os três rankings sejam idênticos.

Uma divergência entre métricas é justamente um convite para perguntar **qual característica dos documentos está produzindo a diferença**.

In [ ]:
ranking_com_posicoes = ranking.copy()

ranking_com_posicoes["posição Jaccard"] = (
    ranking["Jaccard"]
    .rank(ascending=False, method="min")
)

ranking_com_posicoes["posição contagens"] = (
    ranking["cosseno contagens"]
    .rank(ascending=False, method="min")
)

ranking_com_posicoes["posição TF-IDF"] = (
    ranking["cosseno TF-IDF"]
    .rank(ascending=False, method="min")
)

ranking_com_posicoes["amplitude de posições"] = (
    ranking_com_posicoes[
        ["posição Jaccard", "posição contagens", "posição TF-IDF"]
    ].max(axis=1)
    -
    ranking_com_posicoes[
        ["posição Jaccard", "posição contagens", "posição TF-IDF"]
    ].min(axis=1)
)

display(
    ranking_com_posicoes
    .sort_values("amplitude de posições", ascending=False)
    .head(10)
    .round(3)
)

## 4. Inspecionar documentos semelhantes e casos divergentes

Uma medida de similaridade produz números; a interpretação exige voltar aos documentos.

É útil selecionar dois tipos de caso:

1. documentos que aparecem no topo de várias métricas;
2. documentos cuja posição muda bastante entre as métricas.

O primeiro caso mostra concordância entre representações. O segundo ajuda a compreender **o que cada medida valoriza**.

Antes de afirmar que dois documentos são semelhantes, devemos observar também seus metadados: `ano`, `genero`, `local` e `tema`.

In [ ]:
top_jaccard = similaridades_jaccard.index[0]
top_tfidf = similaridades_cosseno_tfidf.index[0]
mais_divergente = (
    ranking_com_posicoes["amplitude de posições"]
    .idxmax()
)

ids_inspecao = list(dict.fromkeys([
    consulta,
    top_jaccard,
    top_tfidf,
    mais_divergente,
]))

print("Documentos selecionados:", ids_inspecao)

display(
    ranking.loc[
        [i for i in ids_inspecao if i != consulta]
    ].round(3)
)

display(
    dados_indexados.loc[
        ids_inspecao,
        ["ano", "genero", "local", "tema", "texto"],
    ]
)

A inspeção permite formular perguntas mais interessantes do que simplesmente “qual documento é o mais parecido?”:

- a proximidade decorre principalmente do `tema`?
- documentos de gêneros diferentes podem ser lexicalmente próximos?
- termos associados ao `local` estão alterando a comparação?
- o TF-IDF muda o ranking porque destaca palavras raras?
- dois documentos do mesmo grupo de metadados são necessariamente os mais semelhantes?

A última pergunta retoma um ponto central da Unidade 5: **agrupamento por metadados e proximidade textual são propriedades diferentes**.

## 5. Distância de edição e comparação de versões

Jaccard e cosseno são adequados para comparar perfis lexicais. Mas, quando temos duas **versões do mesmo documento**, a ordem das unidades pode ser decisiva.

A distância de Levenshtein mede o menor número de:

- inserções;
- exclusões;
- substituições

necessário para transformar uma sequência em outra.

Para as posições $i$ e $j$:

$$
D_{i,j}
=
\min
\begin{cases}
D_{i-1,j}+1 & \text{exclusão}\\
D_{i,j-1}+1 & \text{inserção}\\
D_{i-1,j-1}+\mathbf{1}(a_i\neq b_j) & \text{substituição}.
\end{cases}
$$

A unidade da sequência pode ser caractere ou token. Aqui utilizaremos **tokens**, porque queremos medir alterações lexicais.

A distância bruta tende a crescer com o tamanho do texto. Por isso também calcularemos:

$$
D_{\mathrm{norm}}
=
\frac{D}{\max(n_A,n_B)}.
$$

Essa normalização facilita comparações entre pares de comprimentos diferentes.

In [ ]:
def levenshtein(seq_a, seq_b):
    anterior = list(range(len(seq_b) + 1))

    for i, a in enumerate(seq_a, 1):
        atual = [i]

        for j, b in enumerate(seq_b, 1):
            atual.append(min(
                anterior[j] + 1,            # exclusão
                atual[j - 1] + 1,           # inserção
                anterior[j - 1] + (a != b), # substituição
            ))

        anterior = atual

    return anterior[-1]

### Criando versões didáticas para alguns documentos

O conjunto ampliado não contém, atualmente, pares documentais históricos com versões editoriais. Para manter o exercício sem inventar uma fonte externa, criaremos **versões sintéticas derivadas dos próprios textos**.

Para quatro documentos, produziremos:

- a versão `original`, igual ao texto sintético usado anteriormente;
- a versão `editada`, com pequenas operações controladas de inserção, exclusão ou substituição.

Essas alterações são apenas exemplos didáticos do comportamento da distância de edição.

In [ ]:
ids_versoes = dados["id_documento"].head(4).tolist()

registros_versoes = []

for pos, id_documento in enumerate(ids_versoes):
    texto_original = dados_indexados.loc[id_documento, "texto"]
    tokens_original = tokenizar(texto_original)

    tokens_editados = tokens_original.copy()

    if pos == 0:
        tokens_editados[2] = "debatepublico"
    elif pos == 1:
        tokens_editados.insert(4, "arquivo")
    elif pos == 2:
        if len(tokens_editados) > 5:
            del tokens_editados[5]
    else:
        tokens_editados[3] = "documento"
        tokens_editados.append("registro")

    texto_editado = " ".join(tokens_editados)

    registros_versoes.extend([
        {
            "id_documento": id_documento,
            "tipo_versao": "original",
            "texto": texto_original,
        },
        {
            "id_documento": id_documento,
            "tipo_versao": "editada",
            "texto": texto_editado,
        },
    ])

versoes = pd.DataFrame(registros_versoes)
display(versoes)

In [ ]:
resultados_edicao = []

for id_documento, grupo in versoes.groupby("id_documento"):
    original = grupo.loc[
        grupo["tipo_versao"].eq("original")
    ].iloc[0]

    editada = grupo.loc[
        grupo["tipo_versao"].eq("editada")
    ].iloc[0]

    tokens_a = tokenizar(original["texto"])
    tokens_b = tokenizar(editada["texto"])

    distancia = levenshtein(tokens_a, tokens_b)
    denominador = max(len(tokens_a), len(tokens_b))

    resultados_edicao.append({
        "id_documento": id_documento,
        "distância em tokens": distancia,
        "distância normalizada": (
            distancia / denominador
            if denominador
            else 0.0
        ),
        "tokens original": len(tokens_a),
        "tokens editada": len(tokens_b),
    })

tabela_edicao = pd.DataFrame(resultados_edicao)

display(tabela_edicao.round(3))

A distância informa o **esforço mínimo de transformação segundo as operações definidas**. Ela não informa a importância editorial nem a mudança de sentido.

Uma única substituição pode ser substantivamente decisiva, por exemplo, se trocar um nome próprio, uma data ou uma negação. Várias alterações pequenas podem produzir distância maior sem alterar a interpretação central do texto.

Por isso:

$$
\text{distância de edição}
\neq
\text{distância semântica}.
$$

## 6. Comparar grupos de documentos

Até aqui calculamos similaridade entre pares de documentos. Também podemos representar um **grupo** por um vetor agregado.

Por exemplo, podemos calcular o vetor TF-IDF médio de cada `tema` e comparar esses vetores por cosseno.

Isso responde à pergunta:

> **quão semelhantes são os perfis lexicais médios dos grupos?**

É diferente de juntar todos os textos de cada grupo em um único “megadocumento”. A média dá o mesmo peso inicial a cada documento; a concatenação tende a dar mais influência aos grupos ou documentos mais extensos.

In [ ]:
perfil_tema = (
    matriz_tfidf
    .join(
        dados.set_index("id_documento")[["tema"]]
    )
    .groupby("tema")
    .mean()
)

temas = perfil_tema.index.tolist()

matriz_similaridade_temas = pd.DataFrame(
    index=temas,
    columns=temas,
    dtype=float,
)

for tema_a in temas:
    for tema_b in temas:
        matriz_similaridade_temas.loc[tema_a, tema_b] = cosseno(
            perfil_tema.loc[tema_a],
            perfil_tema.loc[tema_b],
        )

display(matriz_similaridade_temas.round(3))

Essa comparação entre grupos fecha a ligação com a pergunta geral da Unidade 5: agrupamentos diferentes podem revelar propriedades diferentes do mesmo conjunto.

Mesmo quando dois grupos apresentam alta similaridade média, isso não significa que todos os seus documentos sejam semelhantes entre si. A heterogeneidade interna continua existindo e deve ser inspecionada.

## 7. Síntese: representação, medida e interpretação

Podemos resumir o notebook como:

$$
\text{pergunta}
\rightarrow
\text{representação}
\rightarrow
\text{medida}
\rightarrow
\text{ranking ou distância}
\rightarrow
\text{inspeção dos casos}
\rightarrow
\text{interpretação}.
$$

As principais distinções são:

- Jaccard mede compartilhamento de vocabulário;
- cosseno de contagens mede semelhança de perfis de frequência;
- cosseno TF-IDF destaca perfis ponderados pela raridade dos termos;
- Levenshtein mede transformação sequencial entre versões;
- nenhuma métrica, sozinha, fornece uma interpretação histórica ou semântica completa.

A divergência entre métricas não é um defeito: ela pode revelar que diferentes propriedades dos documentos estão sendo comparadas.

## U05-A05 — Atividade integrada — relatório de pares

**Modalidade:** trios. **Tempo:** 35 minutos.

1. escolha um documento-consulta e três candidatos;
2. calcule Jaccard, cosseno de contagens e cosseno TF-IDF;
3. identifique pelo menos uma concordância e uma divergência entre os rankings;
4. inspecione os textos e os metadados dos documentos envolvidos;
5. explique qual propriedade textual parece produzir a divergência;
6. escolha um par de versões e calcule a distância de edição;
7. descreva as alterações encontradas sem confundir distância de edição com mudança de sentido;
8. justifique qual medida responde melhor à sua pergunta.

**Documento-consulta e candidatos:** Escreva aqui.

**Métricas e resultados:** Escreva aqui.

**Concordância entre rankings:** Escreva aqui.

**Divergência entre rankings:** Escreva aqui.

**Leitura dos documentos e metadados:** Escreva aqui.

**Comparação de versões:** Escreva aqui.

**Escolha da medida e justificativa:** Escreva aqui.

**O que a similaridade não demonstra:** Escreva aqui.

Leve o relatório à oficina final. A análise deverá explicitar uma medida principal, uma alternativa de sensibilidade e casos que qualifiquem a conclusão.